# 01.03 — The Bias-Variance Tradeoff

**Module:** 01 — ML Fundamentals  
**Prerequisites:** 01.01–01.02  
**Difficulty:** introductory / foundational  
**Estimated time:** 50–70 minutes

This lesson uses synthetic, dimensionless teaching values. It introduces bias, variance, underfitting, overfitting, and irreducible noise without making real-world performance, causal, fairness, or production claims.

**Outcomes:** distinguish training from held-out error; compute residuals and MSE; describe systematic error and prediction sensitivity; explain bias and variance as statistical terms; state the expected squared-error decomposition and its assumptions; and identify why one frozen prediction table cannot estimate variance.


## 1. The Problem

Why can a rule that fits observed examples extremely closely still behave poorly on new inputs? Training error describes the supplied examples; held-out error describes a separate fixed table. A gap can be a diagnostic symptom of instability, but one toy table does not prove a generalisation failure.


## 2. Toy Example or Dataset

We use the synthetic signal `f(x)=x²−x+2`. The observed targets in D1 and D2 add a fixed `+0.5` offset; the test targets use the noiseless signal. The inputs and targets are dimensionless.

| table | inputs | targets |
|---|---|---|
| D1 | `[1, 2, 3, 4]` | `[2.5, 4.5, 8.5, 14.5]` |
| D2 | `[1.5, 2.5, 3.5]` | `[3.75, 6.75, 10.75]` |
| test | `[0.5, 1.25, 2.25]` | `[1.75, 2.3125, 4.8125]` |

The fixed prediction tables are: constant `7`; quadratic `[2,4,8,14]`, `[3.25,6.25,10.25]`, `[1.75,2.3125,4.8125]`; and degree-5 `[2.5,4.5,8.5,14.5]`, `[3.25,6.25,10.25]`, `[3.25,6.25,10.25]`. These are supplied rules, not fitted by this lesson.


In [ ]:
import numpy as np

y_d1 = np.array([2.5, 4.5, 8.5, 14.5], dtype=np.float64)
y_d2 = np.array([3.75, 6.75, 10.75], dtype=np.float64)
y_test = np.array([1.75, 2.3125, 4.8125], dtype=np.float64)


## 3. Intuition Before Math

- **Bias** is statistical systematic error, not social prejudice. A rigid rule can miss the signal in a consistent way.
- **Variance** describes sensitivity of predictions to the training dataset. It is not the variance of the targets and cannot be measured from one fitted table.
- **Underfitting** is a useful description for a rule too rigid to represent the pattern; **overfitting** describes a rule that follows observed examples but is unstable on new inputs.
- **Irreducible noise** is variation in targets that the available inputs and rule cannot remove. Here the fixed offset is a teaching construction, not a random noise estimate.


## 4. Mathematical Derivation

For a fixed input `x`, write `Y = f(x) + ε`, where `ε` has mean zero and variance `σ²_ε`. Let `f̂_D(x)` be the prediction produced by a learning procedure trained on hypothetical dataset `D`. Expected squared prediction error decomposes as

`E_D,ε[(Y − f̂_D(x))²] = [E_D f̂_D(x) − f(x)]² + E_D[(f̂_D(x) − E_D f̂_D(x))²] + σ²_ε`.

The terms are squared bias, variance, and noise variance. This is a theoretical expectation over training datasets and noise under the stated assumptions. It does not apply unchanged to MAE, and the frozen tables below do not estimate its components.


## 5. Manual Walkthrough

For each row, residual is `target − prediction`, squared loss is `residual²`, and MSE is the mean of the squared losses.

| model / table | squared losses | MSE |
|---|---|---:|
| constant / D1 | `[20.25, 6.25, 2.25, 56.25]` | `21.25` |
| constant / D2 | `[10.5625, 0.0625, 14.0625]` | `8.2291666667` |
| constant / test | `[27.5625, 21.97265625, 4.78515625]` | `18.1067708333` |
| quadratic / D1 | `[0.25, 0.25, 0.25, 0.25]` | `0.25` |
| quadratic / D2 | `[0.25, 0.25, 0.25]` | `0.25` |
| quadratic / test | `[0, 0, 0]` | `0` |
| degree-5 / D1 | `[0, 0, 0, 0]` | `0` |
| degree-5 / D2 | `[0.25, 0.25, 0.25]` | `0.25` |
| degree-5 / test | `[2.25, 15.50390625, 29.56640625]` | `15.7734375` |


## 6. Teaching Implementation

The implementation uses explicit float64 arrays and computes pointwise squared losses before averaging. No random sampling, estimator, optimizer, or hidden model-selection routine is used.


In [ ]:
def squared_losses(targets, predictions):
    if targets.shape != predictions.shape:
        raise ValueError("targets and predictions must have matching shapes")
    return (targets - predictions) ** 2

def mean_squared_error(targets, predictions):
    losses = squared_losses(targets, predictions)
    if losses.size == 0:
        raise ValueError("cannot calculate MSE for an empty dataset")
    return float(np.mean(losses))

predictions = {
    "constant": (np.full(4, 7.0), np.full(3, 7.0), np.full(3, 7.0)),
    "quadratic": (np.array([2.0, 4.0, 8.0, 14.0]), np.array([3.25, 6.25, 10.25]), np.array([1.75, 2.3125, 4.8125])),
    "degree5": (np.array([2.5, 4.5, 8.5, 14.5]), np.array([3.25, 6.25, 10.25]), np.array([3.25, 6.25, 10.25])),
}
datasets = ((y_d1, 0), (y_d2, 1), (y_test, 2))
summary = {}
for name, model_predictions in predictions.items():
    summary[name] = tuple(mean_squared_error(targets, model_predictions[index]) for targets, index in datasets)
summary


## 7. Verification

The frozen values, one-dimensional shapes, float64 dtype, non-negative losses, and MSE targets are checked with `rtol=0.0` and `atol=1e-12`.


In [ ]:
RTOL = 0.0
ATOL = 1e-12
expected_mse = {
    "constant": (21.25, 8.229166666666667, 18.106770833333332),
    "quadratic": (0.25, 0.25, 0.0),
    "degree5": (0.0, 0.25, 15.7734375),
}
for targets in (y_d1, y_d2, y_test):
    assert targets.ndim == 1 and targets.dtype == np.float64
for name, values in summary.items():
    np.testing.assert_allclose(values, expected_mse[name], rtol=RTOL, atol=ATOL)
    for targets, index in datasets:
        assert np.all(squared_losses(targets, predictions[name][index]) >= 0.0)
print("verification: PASS")


## 8. Independent Cross-Check

Recompute each residual and squared loss with scalar Python arithmetic. This is deliberately separate from the vectorised functions above. The standalone `verify_hand_calc.py` repeats the same contract without importing this notebook or the private implementation.


In [ ]:
for name, model_predictions in predictions.items():
    for targets, index in datasets:
        scalar_losses = []
        for target, prediction in zip(targets.tolist(), model_predictions[index].tolist(), strict=True):
            residual = target - prediction
            scalar_losses.append(residual ** 2)
        np.testing.assert_allclose(
            scalar_losses, squared_losses(targets, model_predictions[index]), rtol=RTOL, atol=ATOL
        )
print("independent scalar cross-check: PASS")


## 9. Visual Explanation

![Synthetic bias-variance visual](assets/bias_variance_tradeoff.png)

**Caption:** Synthetic, dimensionless instructional data only. The left panel compares frozen predictions with D1 targets; the right compares MSE across D1, D2, and held-out test tables. The degree-5 result is held-out instability, not a variance estimate.

The direct labels, line styles, markers, and hatches provide redundancy so the figure does not rely on colour alone.


## 10. Failure Modes

- Do not use “bias” to mean prejudice.
- Do not call target spread variance of a learning procedure.
- Do not infer a theoretical variance component from one prediction table.
- Do not treat a train/held-out gap as definitive proof of overfitting.
- Do not assume more complexity is always better.
- Do not apply the displayed decomposition unchanged to MAE.
- Do not generalise this synthetic example to production or causal claims.


## 11. Exercises

1. Which rule has the largest D1 MSE, and what systematic pattern do its residuals show?
2. Calculate the degree-5 test MSE from its three squared losses.
3. What additional information would be needed to estimate prediction variance?
4. Explain why the quadratic rule’s test MSE is zero here without claiming that every future dataset will have zero error.


## 12. References

- Hastie, Tibshirani, and Friedman, *The Elements of Statistical Learning*, 2nd ed., §7.3, https://hastie.su.domains/ElemStatLearn/
- James, Witten, Hastie, and Tibshirani, *An Introduction to Statistical Learning*, 2nd ed., Chapter 2, https://www.statlearning.com/
- NumPy documentation, `numpy.testing.assert_allclose`, https://numpy.org/doc/stable/reference/generated/numpy.testing.assert_allclose.html
